# Cross-family activation stitching — the universality test

**Recipient FIXED**: `google/gemma-2-2b` at `L_R = 20`.
**Donor VARIES by family** (flip `DONOR_ARM` in CELL 2 and re-run the whole notebook):

| arm | donor | start `L_D` | also sweep |
|-----|-------|-------------|------------|
| A `"qwen"`  | `Qwen/Qwen2.5-7B`      | 23 | 13 / 18 / 25 |
| B `"llama"` | `meta-llama/Llama-3.1-8B` | 17 | 23 / 28 (v1: Llama's answer crystallizes deep) |

The within-family control (`gemma-2-9b -> gemma-2-2b`) already exists elsewhere and is NOT rebuilt here.

### What is architecturally new: TWO TOKENIZERS
Every earlier notebook assumed donor and recipient share a tokenizer. They do not here. Consequences,
all handled below:

1. `tokenizer_d` and `tokenizer_r` are loaded separately. Helpers that used a global `tokenizer` are
   replaced by explicitly parameterized versions in **CELL X0**.
2. The same-tokenizer assert of CELL 6 is **deleted** (it would fire and is not applicable).
3. Each model gets its **own** encoding of the same problem text. The graft site is each model's own
   **last prompt token** = the position whose logits predict the first answer token. That position is
   well defined in each model independently; only ONE position is grafted, so no token alignment is needed.
4. **Scoring is tokenizer-agnostic.** No token ID is ever compared across models. Everything is scored on
   DECODED generated text: (a) full-answer correctness, (b) leading-digit correctness. The old
   "first-token conferral" metric is not valid cross-family and has been removed as a headline number.
5. Donor-solved and recipient-unsolvable bins are each defined by that model's own tokenization + its own
   decoded-answer check.

Tasks: arithmetic `a * b + c` (workhorse) and symbolic binding chains (colour words are single tokens in
essentially every BPE vocab, so they sidestep tokenization mismatch). No state tracking, no GSM8K.

Run top-to-bottom. **ONE kernel restart after CELL 1.**

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) ===
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
!pip uninstall -y torchvision torchaudio
# torchvision removed before any import (version mismatch crashes transformers). RESTART after this.
# >>> RESTART THE KERNEL NOW, then run every cell below in order. <<<
# --- Blackwell/sm_120 pods ONLY (verify cell prints cap (12,0)): uncomment, run, restart again ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


In [1]:
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")

torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)


In [2]:
# === CELL 2: imports, set_submodule shim, global config (CROSS-FAMILY donor-arm switch) ===
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)

# ---------------- CROSS-FAMILY CONFIG ----------------
# RECIPIENT is fixed; DONOR varies by family. Flip DONOR_ARM and re-run the notebook end to end.
MODEL_R = "google/gemma-2-2b"     # recipient (fixed)
L_R     = 20                      # recipient graft layer (validated in the within-family work)

DONOR_ARM = "qwen"                # "qwen" (ARM A)  or  "llama" (ARM B)
if DONOR_ARM == "qwen":
    MODEL_D = "Qwen/Qwen2.5-7B"          # 28 layers, d_model 3584
    L_D = 23                             # donor graft-layer start guess
    DONOR_SWEEP_LAYERS = [13, 18, 23, 25]
elif DONOR_ARM == "llama":
    MODEL_D = "meta-llama/Llama-3.1-8B"  # 32 layers, d_model 4096
    L_D = 17                             # donor graft-layer start guess
    # v1 found Llama's answer crystallizes DEEP, so the sweep must reach 23/28.
    DONOR_SWEEP_LAYERS = [17, 23, 28]
else:
    raise ValueError("DONOR_ARM must be 'qwen' or 'llama'")

PATCH_POS = -1                    # graft site = last prompt position (right-aligned by left padding)
RIDGE_LAMBDA = 1e3

# ---------------- experiment switches ----------------
RUN_LAYER_SWEEP = True    # EXP 1: donor-layer derivation (map sweep + per-layer donor probe)
RUN_CORE        = True    # EXP 2: recon / task(5 seeds) / SHUFFLE / self-graft
RUN_CEIL        = True    # EXP 3: per-example free-vector ceiling (capacity control)
RUN_MLP         = True    # EXP 4: nonlinear (MLP) map (capacity control)
RUN_TRANSCRIBE  = True    # EXP 5: leading-digit probe on donor state and on the stitched vector
RUN_ABLATE      = True    # EXP 6: INLP answer-subspace erasure + matched-rank random control
RUN_SAE         = True    # EXP 7: recipient-side GemmaScope feature delta at L_R
RUN_SYMBIND     = True    # EXP 8: symbolic binding chains, same channel battery

# Smoke test lever (True for a ~10 min integration check, False for the real run)
SMOKE_TEST = False

if SMOKE_TEST:
    N_ARITH_TRAIN, N_ARITH_EVAL = 200, 150
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
    CEIL_N, CEIL_STEPS = 24, 30
    G_INLP_ROUNDS = 8
    SYMBIND_DEPTHS = [1, 3]
    N_SYMBIND_TRAIN_PER_DEPTH, N_SYMBIND_EVAL_PER_DEPTH = 60, 40
else:
    N_ARITH_TRAIN, N_ARITH_EVAL = 3000, 2000     # -> a few hundred to ~1k in the unsolvable bin
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6 # 5 seeds for the task map
    BOOT_B = 10000
    CEIL_N, CEIL_STEPS = 200, 300                # free-vector ceiling is capped for runtime
    G_INLP_ROUNDS = 60
    SYMBIND_DEPTHS = [1, 3, 5]
    N_SYMBIND_TRAIN_PER_DEPTH, N_SYMBIND_EVAL_PER_DEPTH = 500, 350

SYMBIND_SEEDS = [0]           # channel battery on symbind uses one seed (recon/task/shuffle contrast)
ARITH_BATCH, MAX_NEW_ARITH = 16, 8
MAX_NEW_SYMBIND = 4
RESULTS = {}   # everything defensible gets concentrated here and printed/saved at the end
print("DONOR_ARM =", DONOR_ARM, "| donor =", MODEL_D, "L_D =", L_D,
      "| recipient =", MODEL_R, "L_R =", L_R, "| SMOKE_TEST =", SMOKE_TEST)

DONOR_ARM = qwen | donor = Qwen/Qwen2.5-7B L_D = 23 | recipient = google/gemma-2-2b L_R = 20 | SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Gemma and Llama are gated) ===
from huggingface_hub import login
login("")   # <-- paste your own read token here before running

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, full-answer) ===
# Copied VERBATIM from the base notebook. NOTE: `states_and_top` and `arith_fullanswer_correct`
# below close over a global `tokenizer`; CELL 6 aliases `tokenizer = tokenizer_r` so they remain
# correct if called, but the CROSS-FAMILY code in this notebook uses the explicitly
# tokenizer-parameterized replacements defined in CELL X0 instead.
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

import re as _re
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")

helpers defined


In [7]:
# === CELL X0: CROSS-FAMILY (dual-tokenizer) helpers — run right after CELL 5 ===
# Everything here exists because donor and recipient DO NOT share a tokenizer.
#
# DESIGN DECISION 1 — one problem, two encodings.
#   Each problem dict carries ids_d/tok_d (donor tokenization) and ids_r/tok_r (recipient
#   tokenization) of the SAME text. `ids_*` ends at that model's own LAST PROMPT TOKEN, i.e. the
#   position whose logits predict the first answer token. On Gemma that is the standalone leading
#   space; on Qwen/Llama the space fuses with the first digit so it is the "=" token. Different
#   surface positions, identical operative role. Only this ONE position is grafted, so the two
#   token sequences never need to align.
#
# DESIGN DECISION 2 — scoring is tokenizer-agnostic.
#   No token ID is compared across models anywhere. gen_score_arith greedily generates from the
#   recipient and scores the DECODED text: full-answer correctness and leading-digit correctness.
#   The old "first-token conferral" metric is NOT valid cross-family and is not reported as a
#   headline. `tok_r` is still used, but only as the CE target when training the map — that is a
#   purely within-recipient quantity in the recipient's own vocabulary, which is legitimate.
import torch, torch.nn.functional as F, numpy as np, random, json

def fd(x):
    """First (leading) digit of an integer answer, 0-9. Defined here because the notebooks this
    is assembled from define it only in cells we do not include."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

def probe_first_digit(Xtr, ytr, Xte, yte, steps=300, nclass=10, lr=1e-2):
    """Linear probe on a hidden state -> class label. Returns PREDICTIONS on Xte."""
    Pw = torch.zeros(Xtr.shape[1], nclass, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=lr); mu = Xtr.mean(0); A, Bx = Xtr-mu, Xte-mu
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(A @ Pw, ytr).backward(); opt.step()
    return (Bx @ Pw.detach()).argmax(1)

def probe_split_acc(X, y, nclass=10):
    """Half/half split probe accuracy as a Wilson-CI string. X, y are CPU tensors."""
    if len(y) < 40: return "n/a (n<40)"
    h = len(y)//2
    pred = probe_first_digit(X[:h].float(), y[:h], X[h:].float(), y[h:], nclass=nclass)
    return fmt(wilson_bools((pred == y[h:]).tolist()))

# ---- tokenizer-parameterized state collection (replaces CELL 5's states_and_top) ----
@torch.inference_mode()
def states_and_top_tok(model, tok, layers, prob_ids, batch=None):
    """Left-pad with THIS model's own pad id, forward once, take the last-position residual at
    each requested layer. Returns ({layer: [N,d]}, top_token_ids). The top ids are a diagnostic
    only; they are never compared across models."""
    batch = batch or ARITH_BATCH
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tok.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- arithmetic data with BOTH encodings ----
def gen_arith_dual(n, rng, exclude=None):
    """Generate a*b+c problems that BOTH tokenizers can encode, so donor and recipient see an
    identical problem set. Drops a problem if either tokenizer is not prefix-consistent."""
    exclude = exclude or set(); out, seen = [], set(); tries = 0; dropped = 0
    while len(out) < n and tries < n*200:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids_d, tid_d = _aencode(tokenizer_d, expr, ans)
        ids_r, tid_r = _aencode(tokenizer_r, expr, ans)
        if tid_d is None or tid_r is None:
            dropped += 1; continue
        out.append(dict(expr=expr, ans=ans, ids_d=ids_d, tok_d=tid_d, ids_r=ids_r, tok_r=tid_r))
    if dropped: print(f"  gen_arith_dual: dropped {dropped} problems (tokenizer prefix-inconsistent)")
    return out

# ---- THE scoring function: decoded-answer metrics only ----
@torch.inference_mode()
def gen_score_arith(model, tok, layer, probs, ids_key, vecs=None, batch=None, max_new=None):
    """Greedy-generate from `model` using its OWN encoding (probs[i][ids_key]) and score the
    DECODED text. If vecs is given, vecs[i] is grafted at the last prompt position during prefill.
    Returns {"full": [bool], "lead": [bool]} -- (a) whole-answer string correctness,
    (b) leading-digit correctness. Tokenizer-agnostic by construction."""
    batch = batch or ARITH_BATCH; max_new = max_new or MAX_NEW_ARITH
    full, lead, handle = [], [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p[ids_key] for p in chunk], tok.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=max_new,
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            txt = tok.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                full.append(pred is not None and abs(pred - p["ans"]) < 0.5)
                lead.append(pred is not None and fd(pred) == fd(p["ans"]))
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return {"full": full, "lead": lead}

def score_pair(sc):
    """Format a {'full':..,'lead':..} score dict as two Wilson-CI strings."""
    return {"full": fmt(wilson_bools(sc["full"])), "lead": fmt(wilson_bools(sc["lead"]))}

print("cross-family helpers defined (fd, probe_first_digit, probe_split_acc, "
      "states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)")

cross-family helpers defined (fd, probe_first_digit, probe_split_acc, states_and_top_tok, gen_arith_dual, gen_score_arith, score_pair)


In [8]:
# === CELL 6: load BOTH models and BOTH tokenizers; donor in bf16 (NOT 4-bit) ===
# CHANGE vs every earlier notebook: two separate tokenizers, and the same-tokenizer assert that
# used to live at the bottom of this cell is DELETED — it would fire on every cross-family pair
# and is not applicable, because we graft exactly one position that each model locates for itself.
tokenizer_r = AutoTokenizer.from_pretrained(MODEL_R)     # RECIPIENT tokenizer (gemma-2-2b)
tokenizer_d = AutoTokenizer.from_pretrained(MODEL_D)     # DONOR tokenizer (Qwen / Llama)
for _t in (tokenizer_r, tokenizer_d):
    _t.padding_side = "left"
    if _t.pad_token is None: _t.pad_token = _t.eos_token
# Back-compat alias so any verbatim-copied CELL 5 helper that references a bare `tokenizer`
# operates on the RECIPIENT (all generation in this notebook happens on the recipient).
tokenizer = tokenizer_r

# QUANTIZE_DONOR: keep False. Under 4-bit this transformers/bnb build runs the unquantized layers
# in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) -> NaN logits -> argmax collapses to
# token 0. bf16 has the exponent range to avoid this. A 7-8B donor is ~15-16 GB in bf16, which
# fits alongside the 5 GB recipient on a 48 GB card with room to spare.
QUANTIZE_DONOR = globals().get("QUANTIZE_DONOR", False)

model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_DONOR:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_d = AutoModelForCausalLM.from_pretrained(
        MODEL_D, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_d = AutoModelForCausalLM.from_pretrained(
        MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()

N_LAYERS_D = model_d.config.num_hidden_layers
N_LAYERS_R = model_r.config.num_hidden_layers
D_DIM = model_d.config.hidden_size
R_DIM = model_r.config.hidden_size
print(f"loaded recipient {MODEL_R}: {N_LAYERS_R} layers, d={R_DIM}")
print(f"loaded donor     {MODEL_D}: {N_LAYERS_D} layers, d={D_DIM} | "
      f"{'4-bit' if QUANTIZE_DONOR else 'bf16'}")
assert 0 <= L_R < N_LAYERS_R, f"L_R={L_R} out of range for recipient"
assert 0 <= L_D < N_LAYERS_D, f"L_D={L_D} out of range for donor"
DONOR_SWEEP_LAYERS = sorted({L for L in DONOR_SWEEP_LAYERS if 0 <= L < N_LAYERS_D} | {L_D})
print("donor sweep layers:", DONOR_SWEEP_LAYERS)

# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# hundreds of silently-filtered problems later. A healthy donor tops a real word here.
with torch.inference_mode():
    _hl = model_d(tokenizer_d("The capital of France is",
                              return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "donor produced NaN/Inf logits — numerical blowup. If QUANTIZE_DONOR=True, the donor is "
    "overflowing fp16; set QUANTIZE_DONOR=False to load it in bf16 (needs the VRAM but is safe).")
print("donor health check ok; donor top token:", repr(tokenizer_d.decode([_hl.argmax().item()])))
del _hl

# Show the tokenizer divergence explicitly — this is the whole reason for CELL X0.
_probe = "3 * 12 + 7 = 43"
print("recipient ids:", tokenizer_r(_probe).input_ids)
print("donor     ids:", tokenizer_d(_probe).input_ids)
print("NOTE: sequences differ. That is expected and fine — only ONE position is grafted, and each "
      "model locates that position (its own last prompt token) independently.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

loaded recipient google/gemma-2-2b: 26 layers, d=2304
loaded donor     Qwen/Qwen2.5-7B: 28 layers, d=3584 | bf16
donor sweep layers: [13, 18, 23, 25]
donor health check ok; donor top token: ' Paris'
recipient ids: [2, 235304, 649, 235248, 235274, 235284, 963, 235248, 235324, 589, 235248, 235310, 235304]
donor     ids: [18, 353, 220, 16, 17, 488, 220, 22, 284, 220, 19, 18]
NOTE: sequences differ. That is expected and fine — only ONE position is grafted, and each model locates that position (its own last prompt token) independently.


In [9]:
# === CELL X1: arithmetic data, DONOR-SOLVED and RECIPIENT-UNSOLVABLE bins, reconstruction map ===
# Both filters are decoded-answer checks on each model's OWN tokenization. No token IDs cross the
# family boundary. The unsolvable bin = {donor gets it right} AND {recipient gets it wrong}, so any
# success after the graft was conferred by the stitch.
train = gen_arith_dual(N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith_dual(N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

# ---- donor states + DONOR-SOLVED filter (donor's own encoding, donor's own decoded answer) ----
Xd_t_all, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in train])
X_dt = Xd_t_all[L_D]
Xd_e_all, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in evalp])
X_de_raw = Xd_e_all[L_D]
donor_sc = gen_score_arith(model_d, tokenizer_d, L_D, evalp, "ids_d", vecs=None)
keep = [i for i in range(len(evalp)) if donor_sc["full"][i]]
print(f"  donor solves {len(keep)}/{len(evalp)} "
      f"(full {fmt(wilson_bools(donor_sc['full']))}, lead {fmt(wilson_bools(donor_sc['lead']))})")
evalp = [evalp[i] for i in keep]; X_de = X_de_raw[keep]
del Xd_e_all, X_de_raw

# ---- recipient states + RECIPIENT-UNSOLVABLE filter (recipient's own encoding + decoded answer) ----
Xr_t_all, _ = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in train])
X_rt = Xr_t_all[L_R]
Xr_e_all, r_top = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in evalp])
X_re = Xr_e_all[L_R]
native_sc = gen_score_arith(model_r, tokenizer_r, L_R, evalp, "ids_r", vecs=None)
solvable = native_sc["full"]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv   = [i for i in range(len(evalp)) if solvable[i]]
print(f"  recipient natively solves {len(solv)}/{len(evalp)} -> UNSOLVABLE BIN n={len(unsolv)}")
del Xd_t_all, Xr_t_all, Xr_e_all

# ---- reconstruction map (ridge, donor L_D -> recipient L_R) ----
mu_d, mu_r, Wr = fit_ridge(X_dt, X_rt)
mu_dd, mu_rd = mu_d.to(DEVICE), mu_r.to(DEVICE)
recon_map = (mu_dd, mu_rd, Wr.to(DEVICE))
def map_recon(xd):
    m9, m2, W = recon_map; return (xd.to(DEVICE)-m9) @ W + m2

# ---- the one call every experiment uses to score a graft ----
def confer(vecs, idxs, probs=None):
    """Graft vecs[k] into problem probs[idxs[k]] on the recipient and score DECODED output.
    `vecs` is a [len(idxs), R_DIM] tensor. Returns {'full': [bool], 'lead': [bool]}."""
    probs = evalp if probs is None else probs
    return gen_score_arith(model_r, tokenizer_r, L_R, [probs[j] for j in idxs], "ids_r",
                           vecs=list(vecs.detach().float().cpu()))

# ---- donor leading-digit probe at L_D (always computed; the MLP/ceiling cells reference it) ----
_y_all = torch.tensor([fd(p["ans"]) for p in evalp])
donor_probe_LD = probe_split_acc(X_de, _y_all)
RESULTS["setup"] = {
    "donor": MODEL_D, "recipient": MODEL_R, "L_D": L_D, "L_R": L_R,
    "d_donor": int(X_dt.shape[1]), "d_recipient": int(X_rt.shape[1]),
    "n_train": len(train), "n_eval_donor_solved": len(evalp),
    "n_unsolvable": len(unsolv), "n_solvable": len(solv),
    "donor_full_on_all_eval": fmt(wilson_bools(donor_sc["full"])),
    "recipient_native_full_donor_solved": fmt(wilson_bools(native_sc["full"])),
    "recipient_native_lead_donor_solved": fmt(wilson_bools(native_sc["lead"])),
    "recipient_native_lead_on_unsolv": fmt(wilson_bools([native_sc["lead"][i] for i in unsolv])),
    f"donor_leading_digit_probe_L{L_D}": donor_probe_LD,
    "_note": "all bins/metrics are decoded-answer based; no token id is compared across families",
}
print(json.dumps(RESULTS["setup"], indent=2))

arith: 3000 train, 2000 eval
  donor solves 508/2000 (full 0.254 [0.235, 0.274], lead 0.839 [0.822, 0.854])
  recipient natively solves 69/508 -> UNSOLVABLE BIN n=439
{
  "donor": "Qwen/Qwen2.5-7B",
  "recipient": "google/gemma-2-2b",
  "L_D": 23,
  "L_R": 20,
  "d_donor": 3584,
  "d_recipient": 2304,
  "n_train": 3000,
  "n_eval_donor_solved": 508,
  "n_unsolvable": 439,
  "n_solvable": 69,
  "donor_full_on_all_eval": "0.254 [0.235, 0.274]",
  "recipient_native_full_donor_solved": "0.136 [0.109, 0.168]",
  "recipient_native_lead_donor_solved": "0.604 [0.561, 0.646]",
  "recipient_native_lead_on_unsolv": "0.542 [0.495, 0.588]",
  "donor_leading_digit_probe_L23": "0.606 [0.545, 0.664]",
  "_note": "all bins/metrics are decoded-answer based; no token id is compared across families"
}


In [10]:
# === CELL X2: EXP 1 — DONOR-LAYER DERIVATION (map sweep + per-layer donor answer probe) ===
# Train the SAME task map from DIFFERENT donor layers with the recipient layer fixed at L_R, and
# report the donor's own leading-digit probe at each layer. The law predicts per-layer conferral
# tracks per-layer donor decodability. For Llama this is also the re-site fix: its within-family
# validated donor layer (17) is shallow, while the answer crystallizes around 23-28.
# Based on CELL G2 of Followup2_Gemma.ipynb, rewritten for dual tokenizers + decoded scoring.
if RUN_LAYER_SWEEP and unsolv:
    print("collecting donor states at layers", DONOR_SWEEP_LAYERS, "...")
    Xd_t_sw, _ = states_and_top_tok(model_d, tokenizer_d, DONOR_SWEEP_LAYERS, [p["ids_d"] for p in train])
    Xd_e_sw, _ = states_and_top_tok(model_d, tokenizer_d, DONOR_SWEEP_LAYERS, [p["ids_d"] for p in evalp])
    y_all = torch.tensor([fd(p["ans"]) for p in evalp])
    dsweep = {}
    for Ld in DONOR_SWEEP_LAYERS:
        Xd_t_L, Xd_e_L = Xd_t_sw[Ld], Xd_e_sw[Ld]
        probe = probe_split_acc(Xd_e_L, y_all)
        m9, m2c, Wl = fit_ridge(Xd_t_L, X_rt)
        m9d = m9.to(DEVICE)
        torch.manual_seed(0); random.seed(0)
        W = Wl.clone().to(DEVICE).requires_grad_(True)
        b = m2c.clone().to(DEVICE).requires_grad_(True)
        o = torch.optim.Adam([W, b], lr=1e-3)
        model_r.requires_grad_(False)
        hd = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
        idx = list(range(len(train)))
        try:
            for ep in range(TASK_EPOCHS):
                random.Random(ep).shuffle(idx)
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s+ARITH_BATCH]
                    _graft["vec"] = (Xd_t_L[sub].to(DEVICE) - m9d) @ W + b
                    ids, m = left_pad([train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    # CE target is the RECIPIENT's own first answer token — a within-recipient
                    # quantity in the recipient's vocabulary, so it is valid cross-family.
                    tgt = torch.tensor([train[k]["tok_r"] for k in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt); o.zero_grad(); loss.backward(); o.step()
        finally:
            hd.remove(); _graft["vec"] = None
        model_r.requires_grad_(True)
        W, b = W.detach(), b.detach()
        vecs = (Xd_e_L[unsolv].to(DEVICE) - m9d) @ W + b
        sc = confer(vecs, unsolv)
        dsweep[f"L{Ld}"] = {"donor_probe_leading_digit": probe,
                            "task_confer_unsolv": score_pair(sc),
                            "final_batch_CE": round(float(loss.item()), 3)}
        print(f"  donor L{Ld}: probe={probe}  confer={dsweep[f'L{Ld}']['task_confer_unsolv']}")
    RESULTS["layer_derivation"] = {"recipient_layer": L_R, "by_donor_layer": dsweep}
    del Xd_t_sw, Xd_e_sw
    print("EXP1 donor-layer derivation:", json.dumps(RESULTS["layer_derivation"], indent=2))
else:
    print("EXP1 skipped (RUN_LAYER_SWEEP=False or empty unsolvable bin).")

collecting donor states at layers [13, 18, 23, 25] ...
  donor L13: probe=0.358 [0.302, 0.419]  confer={'full': '0.155 [0.124, 0.192]', 'lead': '0.809 [0.769, 0.843]'}
  donor L18: probe=0.331 [0.276, 0.391]  confer={'full': '0.146 [0.116, 0.182]', 'lead': '0.761 [0.719, 0.798]'}
  donor L23: probe=0.606 [0.545, 0.664]  confer={'full': '0.130 [0.102, 0.165]', 'lead': '0.815 [0.777, 0.849]'}
  donor L25: probe=0.803 [0.750, 0.847]  confer={'full': '0.155 [0.124, 0.192]', 'lead': '0.904 [0.873, 0.928]'}
EXP1 donor-layer derivation: {
  "recipient_layer": 20,
  "by_donor_layer": {
    "L13": {
      "donor_probe_leading_digit": "0.358 [0.302, 0.419]",
      "task_confer_unsolv": {
        "full": "0.155 [0.124, 0.192]",
        "lead": "0.809 [0.769, 0.843]"
      },
      "final_batch_CE": 0.182
    },
    "L18": {
      "donor_probe_leading_digit": "0.331 [0.276, 0.391]",
      "task_confer_unsolv": {
        "full": "0.146 [0.116, 0.182]",
        "lead": "0.761 [0.719, 0.798]"
      }

In [11]:
# === CELL X3: train the task-supervised map at L_D, 5 seeds (the only stochastic part) ===
# Warm-started at the ridge reconstruction map. Trained by CE on the RECIPIENT's own first answer
# token (within-recipient vocabulary -> valid cross-family). Scoring later is decoded-text only.
task_maps = []
model_r.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu_r.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                xd = X_dt[sub].to(DEVICE)
                _graft["vec"] = (xd - mu_dd) @ W + b
                ids, m = left_pad([train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok_r"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}")
model_r.requires_grad_(True)

def map_task(i):
    W, b = task_maps[i]
    return lambda xd: (xd.to(DEVICE)-mu_dd) @ W + b
print(f"trained {len(task_maps)} task maps at donor L{L_D} -> recipient L{L_R}")

  seed 0: final batch CE 0.042
  seed 1: final batch CE 0.344
  seed 2: final batch CE 0.163
  seed 3: final batch CE 0.027
  seed 4: final batch CE 0.111
trained 5 task maps at donor L23 -> recipient L20


In [12]:
# === CELL X4: EXP 2 — CORE CHANNEL: recon / task (5 seeds) / SHUFFLE / self-graft ===
# SHUFFLE is the critical control here. A cross-family map has far more room to learn a generic
# task prior than a within-family one, so if task ~= shuffle, that is the clean NEGATIVE result:
# the map learned "emit a plausible product", not "read this donor's answer".
# Self-graft = re-inject the recipient's OWN native L_R state. It must reproduce native behaviour
# (~0 on the unsolvable bin, by the bin's definition); it is the hook-machinery sanity check and
# proves the graft site itself confers nothing.
# Based on CELLs 8/9/10 of consolidated_eval.ipynb, rewritten for decoded-answer scoring.
if RUN_CORE and unsolv:
    arr = {"n_unsolv": len(unsolv), "n_solv": len(solv)}

    # continuous diagnostic: how close does each map land to the recipient's real state?
    rc_recon = F.cosine_similarity(map_recon(X_de).cpu(), X_re, dim=1).numpy()
    rc_task  = F.cosine_similarity(map_task(0)(X_de).detach().cpu(), X_re, dim=1).numpy()
    arr["recon_cos_reconmap"] = fmt(bootstrap_ci(rc_recon))
    arr["recon_cos_taskmap"]  = fmt(bootstrap_ci(rc_task))

    # ---- headline bin: UNSOLVABLE ----
    arr["native_unsolv"]  = {"full": "0.000 [by construction]",
                             "lead": fmt(wilson_bools([native_sc["lead"][i] for i in unsolv]))}
    arr["selfgraft_unsolv"] = score_pair(confer(X_re[unsolv].to(DEVICE), unsolv))
    arr["recon_unsolv"]     = score_pair(confer(map_recon(X_de[unsolv]), unsolv))

    seed_full, seed_lead = [], []
    for s in range(len(task_maps)):
        sc = confer(map_task(s)(X_de[unsolv]), unsolv)
        seed_full.append(float(np.mean(sc["full"]))); seed_lead.append(float(np.mean(sc["lead"])))
        if s == 0: arr["task_unsolv_seed0"] = score_pair(sc)
    arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
    arr["task_unsolv_lead_acrossseed"] = fmt(across_seed_ci(seed_lead))

    # SHUFFLE: task map fed the WRONG problem's donor state (content-specificity control)
    g = torch.Generator().manual_seed(0)
    shuf = torch.randperm(len(unsolv), generator=g)
    shuf_idx = [unsolv[int(k)] for k in shuf]
    arr["shuffle_unsolv"] = score_pair(confer(map_task(0)(X_de[shuf_idx]), unsolv))

    # ---- sanity bin: SOLVABLE (the graft must not DESTROY what the recipient already knows) ----
    if solv:
        arr["recon_solv"] = score_pair(confer(map_recon(X_de[solv]), solv))
        arr["task_solv"]  = score_pair(confer(map_task(0)(X_de[solv]), solv))
    arr["_read"] = ("task >> shuffle => the stitch transfers donor-specific content. "
                    "task ~= shuffle => the cross-family map only learned a generic task prior.")
    RESULTS["arithmetic_core"] = arr
    print("EXP2 core channel:", json.dumps(arr, indent=2))
else:
    print("EXP2 skipped (RUN_CORE=False or empty unsolvable bin).")

EXP2 core channel: {
  "n_unsolv": 439,
  "n_solv": 69,
  "recon_cos_reconmap": "0.970 [0.968, 0.972]",
  "recon_cos_taskmap": "0.253 [0.249, 0.258]",
  "native_unsolv": {
    "full": "0.000 [by construction]",
    "lead": "0.542 [0.495, 0.588]"
  },
  "selfgraft_unsolv": {
    "full": "0.005 [0.001, 0.016]",
    "lead": "0.542 [0.495, 0.588]"
  },
  "recon_unsolv": {
    "full": "0.043 [0.028, 0.067]",
    "lead": "0.544 [0.498, 0.590]"
  },
  "task_unsolv_seed0": {
    "full": "0.130 [0.102, 0.165]",
    "lead": "0.815 [0.777, 0.849]"
  },
  "task_unsolv_full_acrossseed": "0.137 [0.129, 0.145]",
  "task_unsolv_lead_acrossseed": "0.823 [0.807, 0.839]",
  "shuffle_unsolv": {
    "full": "0.021 [0.011, 0.038]",
    "lead": "0.182 [0.149, 0.221]"
  },
  "recon_solv": {
    "full": "0.623 [0.505, 0.728]",
    "lead": "0.797 [0.688, 0.875]"
  },
  "task_solv": {
    "full": "0.493 [0.378, 0.608]",
    "lead": "0.754 [0.640, 0.840]"
  },
  "_read": "task >> shuffle => the stitch transfers d

In [13]:
# === CELL X5: EXP 3 — PER-EXAMPLE FREE-VECTOR CEILING (capacity control) ===
# Removes EVERY constraint on the channel: for each unsolvable problem, optimise an UNCONSTRAINED
# vector injected at the graft site directly against the gold answer SEQUENCE (teacher-forced),
# per example, for many steps. That is the best ANY single-site injection can do. Then FREE-GENERATE
# and score decoded text. If the free vectors also fail, the bottleneck is not map expressiveness or
# the cross-family gap — a single injected vector cannot make the recipient EXECUTE the computation.
# Optimising the WHOLE answer sequence (not just the first token) is what makes this robust to
# tokenization, which matters even more cross-family. Based on EXP14_freevec_ceiling_cell.py.
if RUN_CEIL and unsolv:
    fv_idx = unsolv[:CEIL_N]
    items = [evalp[i] for i in fv_idx]
    seqs, ans_lens, prompt_lens = [], [], []
    for p in items:
        pid  = p["ids_r"].flatten().tolist()                       # RECIPIENT tokenization only
        full_ids = tokenizer_r(_aprompt(p["expr"]) + " " + str(p["ans"])).input_ids
        if full_ids[:len(pid)] != pid:
            full_ids = pid + tokenizer_r(" " + str(p["ans"]), add_special_tokens=False).input_ids
        seqs.append(torch.tensor(full_ids)); prompt_lens.append(len(pid))
        ans_lens.append(len(full_ids) - len(pid))
    Lm, B, pad_id = max(s.numel() for s in seqs), len(seqs), tokenizer_r.pad_token_id
    IDS = torch.full((B, Lm), pad_id, dtype=torch.long)
    TGT = torch.full((B, Lm), -100, dtype=torch.long)
    PE  = torch.zeros(B, dtype=torch.long)
    for i, s in enumerate(seqs):
        n = s.numel(); off = Lm - n                                 # left-pad => right-aligned
        IDS[i, off:] = s
        PE[i] = off + prompt_lens[i] - 1                            # graft site
        TGT[i, PE[i]:PE[i]+ans_lens[i]] = s[prompt_lens[i]:]
    ATT = (IDS != pad_id).long()

    V = map_recon(X_de[fv_idx]).detach().clone().to(DEVICE).float().requires_grad_(True)
    opt = torch.optim.Adam([V], lr=5e-2)
    _ceil = {"V": None, "pe": None}
    def _ceil_hook(_m, _i, o):
        h = _hid(o)
        if _ceil["V"] is None or h.shape[1] <= 1: return o
        h2 = h.clone()
        h2[torch.arange(h.shape[0], device=h.device), _ceil["pe"].to(h.device), :] = _ceil["V"].to(h.dtype)
        return _pack(o, h2)
    model_r.requires_grad_(False)
    handle = model_r.model.layers[L_R].register_forward_hook(_ceil_hook)
    try:
        for step in range(CEIL_STEPS):
            perm = torch.randperm(B); tot = 0.0
            for s in range(0, B, ARITH_BATCH):
                sub = perm[s:s+ARITH_BATCH]
                _ceil["V"] = V[sub]; _ceil["pe"] = PE[sub]
                logits = model_r(IDS[sub].to(DEVICE), attention_mask=ATT[sub].to(DEVICE)).logits.float()
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                       TGT[sub].reshape(-1).to(DEVICE), ignore_index=-100)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()*len(sub)
            if step % 50 == 0 or step == CEIL_STEPS-1:
                print(f"  ceil step {step:3d}  teacher-forced answer CE = {tot/B:.4f}")
    finally:
        handle.remove(); _ceil["V"] = None
    model_r.requires_grad_(True)
    Vf = V.detach()

    sc = confer(Vf, fv_idx)
    out = {"n": B, "steps": CEIL_STEPS, "final_teacherforced_CE": round(float(tot/B), 4),
           "ceiling": score_pair(sc),
           "_ref_task_full": RESULTS.get("arithmetic_core", {}).get("task_unsolv_full_acrossseed", "n/a"),
           "_ref_shuffle":   RESULTS.get("arithmetic_core", {}).get("shuffle_unsolv", "n/a"),
           "_caveat": "teacher forcing gives the vector the gold prefix during training, so the "
                      "free-gen number is a conservative read; it only makes a LOW ceiling more "
                      "credible, not less."}
    RESULTS["freevec_ceiling"] = out
    print("EXP3 free-vector ceiling:", json.dumps(out, indent=2))
else:
    print("EXP3 skipped (RUN_CEIL=False or empty unsolvable bin).")

  ceil step   0  teacher-forced answer CE = 1.4201
  ceil step  50  teacher-forced answer CE = 0.0046
  ceil step 100  teacher-forced answer CE = 0.0010
  ceil step 150  teacher-forced answer CE = 0.0004
  ceil step 200  teacher-forced answer CE = 0.0002
  ceil step 250  teacher-forced answer CE = 0.0001
  ceil step 299  teacher-forced answer CE = 0.0000
EXP3 free-vector ceiling: {
  "n": 200,
  "steps": 300,
  "final_teacherforced_CE": 0.0,
  "ceiling": {
    "full": "1.000 [0.981, 1.000]",
    "lead": "1.000 [0.981, 1.000]"
  },
  "_ref_task_full": "0.137 [0.129, 0.145]",
  "_ref_shuffle": {
    "full": "0.021 [0.011, 0.038]",
    "lead": "0.182 [0.149, 0.221]"
  },
  "_caveat": "teacher forcing gives the vector the gold prefix during training, so the free-gen number is a conservative read; it only makes a LOW ceiling more credible, not less."
}


In [14]:
# === CELL X6: EXP 4 — NONLINEAR (MLP) TASK MAP (capacity control) ===
# Same task-supervised objective, but a nonlinear residual MLP on top of the linear recon map.
# Cross-family is exactly where a linear map is most suspect (different families need not be
# linearly related at all), so this asks whether the linear form is the binding constraint. If a
# nonlinear map still cannot push conferral above the donor probe ceiling, transfer is limited by
# the donor's decodable content, not by the map. Based on CELL 16 of consolidated_eval.ipynb.
if RUN_MLP and unsolv:
    torch.manual_seed(0); random.seed(0)
    d_in, d_out = X_dt.shape[1], X_rt.shape[1]
    mlp = nn.Sequential(nn.Linear(d_in, 1024), nn.GELU(), nn.Linear(1024, d_out)).to(DEVICE)
    nn.init.normal_(mlp[2].weight, std=1e-3); nn.init.zeros_(mlp[2].bias)  # start ~= recon map
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
    model_r.requires_grad_(False)
    handle = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                xd = X_dt[sub].to(DEVICE)
                _graft["vec"] = apply_map(xd, recon_map) + mlp(xd)
                ids, m = left_pad([train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok_r"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    model_r.requires_grad_(True); mlp.eval()
    print(f"  MLP final batch CE {loss.item():.3f}")

    def map_mlp(xd):
        with torch.no_grad():
            xd = xd.to(DEVICE)
            return apply_map(xd, recon_map) + mlp(xd)
    out = {"mlp_unsolv": score_pair(confer(map_mlp(X_de[unsolv]), unsolv)),
           "donor_probe_ceiling_at_L_D": donor_probe_LD,
           "_ref_linear_task_full": RESULTS.get("arithmetic_core", {}).get("task_unsolv_full_acrossseed", "n/a"),
           "_ref_shuffle": RESULTS.get("arithmetic_core", {}).get("shuffle_unsolv", "n/a")}
    RESULTS["nonlinear_task_map"] = out
    print("EXP4 nonlinear task map:", json.dumps(out, indent=2))
else:
    print("EXP4 skipped (RUN_MLP=False or empty unsolvable bin).")

  MLP final batch CE 0.092
EXP4 nonlinear task map: {
  "mlp_unsolv": {
    "full": "0.162 [0.130, 0.199]",
    "lead": "0.831 [0.794, 0.864]"
  },
  "donor_probe_ceiling_at_L_D": "0.606 [0.545, 0.664]",
  "_ref_linear_task_full": "0.137 [0.129, 0.145]",
  "_ref_shuffle": {
    "full": "0.021 [0.011, 0.038]",
    "lead": "0.182 [0.149, 0.221]"
  }
}


In [15]:
# === CELL X7: EXP 5 — TRANSCRIPTION PROBE (leading digit on donor state AND on the stitched vector) ===
# The graft REPLACES the recipient's L_R state with the map output, so "the recipient's state after
# the graft" at that layer IS the map output. Probe the answer's LEADING DIGIT from:
#   native recipient state (no graft) -- should NOT carry the answer on the unsolvable bin,
#   reconstruction-map output, task-map output (the stitched vector), and the donor state (ceiling).
# A large native->task gap = the map WROTE the answer into the recipient's stream (transcription),
# rather than the recipient computing it. Leading digit is used because it is the tokenizer-agnostic
# unit: it is a single token in the recipient's vocabulary regardless of the donor's family.
# Cheap: no model forwards, just linear probes on states already in memory.
# Based on CELL 14 of consolidated_eval.ipynb.
if RUN_TRANSCRIBE:
    pidx = unsolv if len(unsolv) >= 60 else list(range(len(evalp)))
    y_tc = torch.tensor([fd(evalp[i]["ans"]) for i in pidx])
    Xr_sub, Xd_sub = X_re[pidx], X_de[pidx]
    task_out  = map_task(0)(Xd_sub).detach().cpu()
    recon_out = map_recon(Xd_sub).detach().cpu()
    RESULTS["transcription_probe"] = {
        "n": len(pidx), "bin": "unsolvable" if pidx is unsolv else "all_donor_solved",
        f"native_recipient_L{L_R}": probe_split_acc(Xr_sub, y_tc),
        "recon_map_output":         probe_split_acc(recon_out, y_tc),
        "task_map_output_STITCHED": probe_split_acc(task_out, y_tc),
        f"donor_L{L_D}_reference":  probe_split_acc(Xd_sub, y_tc),
        "_read": "native low + stitched ~= donor => the map transcribed the donor's answer; "
                 "stitched ~= native => nothing about the answer crossed the family boundary.",
    }
    print("EXP5 transcription probe:", json.dumps(RESULTS["transcription_probe"], indent=2))
else:
    print("EXP5 skipped (RUN_TRANSCRIBE=False).")

EXP5 transcription probe: {
  "n": 439,
  "bin": "unsolvable",
  "native_recipient_L20": "0.505 [0.439, 0.570]",
  "recon_map_output": "0.550 [0.484, 0.614]",
  "task_map_output_STITCHED": "0.732 [0.670, 0.786]",
  "donor_L23_reference": "0.632 [0.566, 0.693]",
  "_read": "native low + stitched ~= donor => the map transcribed the donor's answer; stitched ~= native => nothing about the answer crossed the family boundary."
}


In [16]:
# === CELL X8: EXP 6 — INLP ANSWER-SUBSPACE ERASURE + MATCHED-RANK RANDOM CONTROL ===
# EXP5 shows the answer is DECODABLE from the stitched vector — correlational. This makes it CAUSAL:
# delete the answer-carrying directions from the grafted vector, re-graft, and see whether conferral
# collapses. The subspace is fit on the TRAIN items only (disjoint from the eval bin we score on).
# Two erasures, each with a MATCHED-RANK RANDOM subspace control:
#   (1) single-shot rank-k from the probe weights   (EXP17_answer_ablation_cell.py)
#   (2) iterative INLP over G_INLP_ROUNDS rounds    (INLP part of CELL G1 in Followup2_Gemma.ipynb)
# Read: answer-ablated -> native floor AND random-ablated -> ~task  ==>  the conferral WAS the
# transcribed answer and nothing else.
if RUN_ABLATE and unsolv:
    base = map_task(0)
    Ttr = base(X_dt).detach()                                  # [N_train, d_r] stitched train vectors
    ytr = torch.tensor([fd(p["ans"]) for p in train], device=DEVICE)

    def _fit_probe_w(X, y, steps=300):
        Pw = torch.zeros(X.shape[1], 10, device=DEVICE, requires_grad=True)
        o = torch.optim.Adam([Pw], lr=1e-2); mu = X.mean(0)
        for _ in range(steps): o.zero_grad(); F.cross_entropy((X-mu)@Pw, y).backward(); o.step()
        return Pw.detach()
    def _proj_out(X, Q): return X if Q is None else X - (X @ Q) @ Q.T
    def _rand_Q(d, r, seed=0):
        torch.manual_seed(seed)
        return torch.linalg.qr(torch.randn(d, r, device=DEVICE)).Q[:, :r]
    def _erased(Qs):
        def f(xd):
            v = base(xd); return v - (v @ Qs) @ Qs.T
        return f

    out = {}
    # ---- (1) single-shot rank-k answer subspace ----
    Pw = _fit_probe_w(Ttr, ytr)
    Pc = Pw - Pw.mean(1, keepdim=True)                          # drop the shift-invariant direction
    U1, S1, _ = torch.linalg.svd(Pc, full_matrices=False)
    k = int((S1 > S1.max()*1e-3).sum().item()); Q1 = U1[:, :k]
    Q1r = _rand_Q(Q1.shape[0], k, seed=0)
    out["oneshot_rank_k"] = k
    out["oneshot_answer_ablated"] = score_pair(confer(_erased(Q1)(X_de[unsolv]), unsolv))
    out["oneshot_random_ablated"] = score_pair(confer(_erased(Q1r)(X_de[unsolv]), unsolv))
    print(f"  one-shot rank {k}: ablated={out['oneshot_answer_ablated']} "
          f"random={out['oneshot_random_ablated']}")

    # ---- (2) iterative INLP ----
    Q = None
    for rnd in range(G_INLP_ROUNDS):
        Pw2 = _fit_probe_w(_proj_out(Ttr, Q), ytr)
        Pc2 = Pw2 - Pw2.mean(1, keepdim=True)
        D = Pc2 if Q is None else Pc2 - Q @ (Q.T @ Pc2)
        Un, Sn, _ = torch.linalg.svd(D, full_matrices=False)
        kp = Un[:, Sn > Sn.max()*1e-3]
        if kp.shape[1] == 0: break
        Q = kp if Q is None else torch.linalg.qr(torch.cat([Q, kp], 1)).Q
        if Q.shape[1] >= Ttr.shape[1] - 1: break
    r = int(Q.shape[1]); Qr = _rand_Q(Q.shape[0], r, seed=1)
    out["inlp_rounds"] = G_INLP_ROUNDS
    out["inlp_rank"] = r
    out["inlp_answer_ablated"] = score_pair(confer(_erased(Q)(X_de[unsolv]), unsolv))
    out["inlp_random_ablated_matched_rank"] = score_pair(confer(_erased(Qr)(X_de[unsolv]), unsolv))
    out["_ref_task_seed0"] = RESULTS.get("arithmetic_core", {}).get("task_unsolv_seed0", "n/a")
    out["_ref_shuffle"]    = RESULTS.get("arithmetic_core", {}).get("shuffle_unsolv", "n/a")
    out["_note_native"]    = "native full-answer on the unsolvable bin is 0 by construction"
    RESULTS["answer_ablation"] = out
    print("EXP6 INLP answer-subspace erasure:", json.dumps(out, indent=2))
else:
    print("EXP6 skipped (RUN_ABLATE=False or empty unsolvable bin).")

  one-shot rank 9: ablated={'full': '0.134 [0.106, 0.169]', 'lead': '0.815 [0.777, 0.849]'} random={'full': '0.139 [0.110, 0.174]', 'lead': '0.818 [0.779, 0.851]'}
EXP6 INLP answer-subspace erasure: {
  "oneshot_rank_k": 9,
  "oneshot_answer_ablated": {
    "full": "0.134 [0.106, 0.169]",
    "lead": "0.815 [0.777, 0.849]"
  },
  "oneshot_random_ablated": {
    "full": "0.139 [0.110, 0.174]",
    "lead": "0.818 [0.779, 0.851]"
  },
  "inlp_rounds": 60,
  "inlp_rank": 540,
  "inlp_answer_ablated": {
    "full": "0.046 [0.030, 0.069]",
    "lead": "0.385 [0.341, 0.431]"
  },
  "inlp_random_ablated_matched_rank": {
    "full": "0.159 [0.128, 0.197]",
    "lead": "0.822 [0.784, 0.855]"
  },
  "_ref_task_seed0": {
    "full": "0.130 [0.102, 0.165]",
    "lead": "0.815 [0.777, 0.849]"
  },
  "_ref_shuffle": {
    "full": "0.021 [0.011, 0.038]",
    "lead": "0.182 [0.149, 0.221]"
  },
  "_note_native": "native full-answer on the unsolvable bin is 0 by construction"
}


In [17]:
# === CELL X9: install sae_lens for the recipient-side SAE (EXP 7) ===
# Run this only if RUN_SAE. Do NOT restart the kernel after it: the already-imported
# transformers 4.46.3 stays live in memory, and the second line restores the pin on disk in case
# sae_lens's dependency resolution bumped it.
!pip install -q sae-lens
!pip install -q --no-deps --force-reinstall "transformers==4.46.3"
print("sae_lens installed; do NOT restart the kernel")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
sae_lens installed; do NOT restart the kernel


In [18]:
# === CELL X10: EXP 7 — RECIPIENT-SIDE SAE FEATURE DELTA at L_R (GemmaScope) ===
# The complementary question to "what does the map discard from the donor": decode the RECIPIENT
# residual with a gemma-2-2b SAE and compare the stitched state (task-map output) to the native
# state. The per-feature delta is exactly what the stitch turns ON/OFF inside the recipient.
# This is the one probe that is unaffected by the family boundary, because it lives entirely on
# the recipient side: gemma-2-2b has full GemmaScope coverage at layer 20 (google/gemma-scope-2b-pt-res).
# Prediction: the stitch injects the recipient's ANSWER features and little else. The recon map is
# the contrast — it should move almost nothing. Based on EXP21_recipient_feature_delta_cell.py.
SAE2B_RELEASE = globals().get("SAE2B_RELEASE", "gemma-scope-2b-pt-res-canonical")
SAE2B_ID      = globals().get("SAE2B_ID", f"layer_{L_R}/width_16k/canonical")
TOPK_DELTA    = globals().get("TOPK_DELTA", 30)

if RUN_SAE and unsolv:
    try:
        from sae_lens import SAE
        sae2 = SAE.from_pretrained(SAE2B_RELEASE, SAE2B_ID, device=str(DEVICE))
        sae2 = sae2[0] if isinstance(sae2, tuple) else sae2
        NF = sae2.W_dec.shape[0]
        print(f"loaded recipient SAE {SAE2B_RELEASE}/{SAE2B_ID}: {NF} features")

        @torch.inference_mode()
        def _enc_sum(vecs2d):
            tot = torch.zeros(NF, device=DEVICE); n = 0
            for s in range(0, vecs2d.shape[0], ARITH_BATCH):
                a = sae2.encode(vecs2d[s:s+ARITH_BATCH].to(DEVICE).float())
                tot += a.sum(0); n += a.shape[0]
            return tot, n

        task_stitch  = map_task(0)(X_de[unsolv]).detach()
        recon_stitch = map_recon(X_de[unsolv]).detach()
        s_task,  n1 = _enc_sum(task_stitch)
        s_recon, _  = _enc_sum(recon_stitch)
        s_nat,   _  = _enc_sum(X_re[unsolv])
        delta_task  = (s_task  - s_nat) / max(n1, 1)
        delta_recon = (s_recon - s_nat) / max(n1, 1)

        # ANSWER features defined INTRINSICALLY: recipient features whose NATIVE activation on
        # SOLVABLE problems tracks the answer's leading digit (eta^2) — independent of the stitch.
        ref_idx = solv if (solv and len(solv) >= 40) else list(range(len(evalp)))
        yv = torch.tensor([fd(evalp[i]["ans"]) for i in ref_idx], device=DEVICE)
        @torch.inference_mode()
        def _eta2(vecs2d, labels):
            gsum = torch.zeros(10, NF, device=DEVICE); gcnt = torch.zeros(10, device=DEVICE)
            tot = torch.zeros(NF, device=DEVICE); tot2 = torch.zeros(NF, device=DEVICE); n = 0
            for s in range(0, vecs2d.shape[0], ARITH_BATCH):
                a = sae2.encode(vecs2d[s:s+ARITH_BATCH].to(DEVICE).float())
                lb = labels[s:s+a.shape[0]]
                gsum.index_add_(0, lb, a)
                gcnt.index_add_(0, lb, torch.ones_like(lb, dtype=torch.float))
                tot += a.sum(0); tot2 += (a*a).sum(0); n += a.shape[0]
            gmean = gsum / gcnt.clamp_min(1).unsqueeze(1); grand = tot / max(n, 1)
            between = (gcnt.unsqueeze(1) * (gmean - grand)**2).sum(0) / max(n, 1)
            total_var = (tot2/max(n, 1) - grand**2).clamp_min(1e-8)
            return between / total_var
        sel = _eta2(X_re[ref_idx], yv)

        top_add  = torch.topk(delta_task, TOPK_DELTA).indices
        top_rem  = torch.topk(-delta_task, TOPK_DELTA).indices
        base_sel = sel.mean().item()
        out = {
            "sae": f"{SAE2B_RELEASE}/{SAE2B_ID}", "n_unsolv": int(len(unsolv)),
            "task_stitch_L1_change":  round(delta_task.abs().sum().item(), 2),
            "recon_stitch_L1_change": round(delta_recon.abs().sum().item(), 2),
            "answer_selectivity_added_features": round(sel[top_add].mean().item(), 4),
            "answer_selectivity_all_features":   round(base_sel, 4),
            "added_over_baseline_ratio": round(sel[top_add].mean().item()/max(base_sel, 1e-8), 2),
            "top_added_feature_ids":   top_add.tolist()[:15],
            "top_removed_feature_ids": top_rem.tolist()[:15],
            "interpretation": "added-feature selectivity >> baseline AND task L1 >> recon L1 => the "
                              "stitch injects the recipient's ANSWER features, not computation.",
        }
        RESULTS["recipient_feature_delta"] = out
        print("EXP7 recipient feature delta:", json.dumps(out, indent=2))
    except Exception as e:
        print("EXP7 needs a gemma-2-2b residual SAE (GemmaScope, google/gemma-scope-2b-pt-res).")
        print("  Fix SAE2B_RELEASE / SAE2B_ID or set RUN_SAE=False. error:", repr(e))
else:
    print("EXP7 skipped (RUN_SAE=False or empty unsolvable bin).")

Error importing huggingface_hub.hf_file_system: cannot import name 'BucketNotFoundError' from 'huggingface_hub.errors' (/usr/local/lib/python3.11/dist-packages/huggingface_hub/errors.py)
EXP7 needs a gemma-2-2b residual SAE (GemmaScope, google/gemma-scope-2b-pt-res).
  Fix SAE2B_RELEASE / SAE2B_ID or set RUN_SAE=False. error: ImportError("cannot import name 'BucketNotFoundError' from 'huggingface_hub.errors' (/usr/local/lib/python3.11/dist-packages/huggingface_hub/errors.py)")


In [19]:
# === CELL X11: EXP 8 — SYMBOLIC BINDING CHAINS, same channel battery, 2-3 depths ===
#   dax is blue. mip is dax. lorn is mip.  What is lorn?
# Alias-chain resolution instead of arithmetic. This task is ESPECIALLY valuable cross-family:
# the answers are colour words, which are single tokens in essentially every BPE vocabulary, so the
# donor/recipient tokenization mismatch is sidestepped almost entirely, and the scoring (parse a
# colour word out of decoded text) is tokenizer-agnostic by construction.
# Adapted from EXP22_symbolic_binding_chains_cell.py / CELL E22 of Followup2_Gemma.ipynb for dual
# tokenizers: every item carries ids_d/tok_d and ids_r/tok_r, donor-solved and recipient-unsolvable
# bins are decoded-label checks on each model's own encoding.
import re
if RUN_SYMBIND:
    SB_LABELS = ["red", "blue", "green", "yellow", "black",
                 "white", "purple", "silver", "gold", "brown"]
    SB_NAMES = ["dax", "wug", "mip", "tav", "lorn", "nix", "pavo", "sarn",
                "keld", "voma", "rusk", "fen", "zup", "bex", "hald", "quim",
                "nora", "vint", "sej", "plin", "marn", "tul", "gref", "dorn",
                "lisk", "pon", "vash", "kib", "zorn", "mel"]
    SB_DISTRACTORS = 2
    SB_FEWSHOT = ("Resolve the final label.\n"
                  "Definitions:\n"
                  "dax is red. wug is dax.\n"
                  "Question: What is wug?\n"
                  "Answer: red\n\n"
                  "Resolve the final label.\n"
                  "Definitions:\n"
                  "mip is blue. tav is mip. lorn is tav.\n"
                  "Question: What is lorn?\n"
                  "Answer: blue\n\n")

    def _sb_prompt(defs, query):
        return (SB_FEWSHOT + "Resolve the final label.\nDefinitions:\n" + " ".join(defs)
                + f"\nQuestion: What is {query}?\nAnswer:")

    def _sb_first_answer_token(tok, prompt, answer):
        """Context ending at THIS tokenizer's last prompt token (the position that predicts the
        first answer token), plus that target token id. Same operative site in both families."""
        p = tok(prompt).input_ids
        f = tok(prompt + " " + answer).input_ids
        if f[:len(p)] == p and len(f) > len(p):
            j = len(p)
            while j < len(f) and tok.decode([f[j]]).strip() == "": j += 1
            if j < len(f): return torch.tensor(f[:j]), f[j]
        suffix = tok(" " + answer, add_special_tokens=False).input_ids   # non-prefix-consistent fallback
        j = 0
        while j < len(suffix) and tok.decode([suffix[j]]).strip() == "": j += 1
        if j >= len(suffix): return None, None
        return torch.tensor(p + suffix[:j]), suffix[j]

    def _sb_make_item(rng, depth, exclude=None):
        names = rng.sample(SB_NAMES, depth + 2*SB_DISTRACTORS)
        label = rng.choice(SB_LABELS); label_id = SB_LABELS.index(label)
        chain = names[:depth]
        defs = [f"{chain[0]} is {label}."]
        for i in range(1, depth): defs.append(f"{chain[i]} is {chain[i-1]}.")
        off = depth; distractor_labels = [x for x in SB_LABELS if x != label]
        for d in range(SB_DISTRACTORS):
            a, b = names[off + 2*d], names[off + 2*d + 1]
            dl = rng.choice(distractor_labels)
            defs.extend([f"{a} is {dl}.", f"{b} is {a}."])
        rng.shuffle(defs)
        prompt = _sb_prompt(defs, chain[-1])
        if exclude and prompt in exclude: return None
        ids_d, tid_d = _sb_first_answer_token(tokenizer_d, prompt, label)
        ids_r, tid_r = _sb_first_answer_token(tokenizer_r, prompt, label)
        if tid_d is None or tid_r is None: return None
        return dict(prompt=prompt, query=chain[-1], answer=label, label=label_id, depth=depth,
                    ids_d=ids_d, tok_d=tid_d, ids_r=ids_r, tok_r=tid_r)

    def gen_symbind(n_per_depth, rng, depths, exclude=None):
        exclude = exclude or set(); out, seen = [], set()
        for depth in depths:
            before = len(out); tries = 0
            while len(out) - before < n_per_depth and tries < n_per_depth*300:
                tries += 1
                item = _sb_make_item(rng, depth, exclude | seen)
                if item is None: continue
                seen.add(item["prompt"]); out.append(item)
            got = len(out) - before
            if got < n_per_depth: print(f"  [symbind depth {depth}] only {got}/{n_per_depth}")
        rng.shuffle(out)
        return out

    def _sb_parse_label(text):
        txt = text.lower().strip()
        m = re.search(r"[a-z]+", txt)
        if m and m.group(0) in SB_LABELS: return m.group(0)
        for lab in SB_LABELS:
            if re.match(rf"^[\s\W]*{re.escape(lab)}\b", txt): return lab
        return None

    @torch.inference_mode()
    def sb_gen_score(model, tok, layer, probs, ids_key, vecs=None, batch=None):
        """Decoded-label scoring. Tokenizer-agnostic: parses a colour word out of the text."""
        batch = batch or ARITH_BATCH
        ok, handle = [], None
        if vecs is not None:
            handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
        try:
            for i in range(0, len(probs), batch):
                chunk = probs[i:i+batch]
                ids, m = left_pad([p[ids_key] for p in chunk], tok.pad_token_id)
                ids, m = ids.to(DEVICE), m.to(DEVICE)
                _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                                 if vecs is not None else None)
                gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_SYMBIND,
                                     do_sample=False, pad_token_id=tok.eos_token_id)
                txt = tok.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
                for p, t in zip(chunk, txt): ok.append(_sb_parse_label(t) == p["answer"])
        finally:
            if handle: handle.remove()
            _graft["vec"] = None
        return ok

    # -------- data, donor-solved filter, recipient-unsolvable filter (all decoded-label) --------
    SB_train = gen_symbind(N_SYMBIND_TRAIN_PER_DEPTH, random.Random(220), SYMBIND_DEPTHS)
    SB_eval_raw = gen_symbind(N_SYMBIND_EVAL_PER_DEPTH, random.Random(221), SYMBIND_DEPTHS,
                              {p["prompt"] for p in SB_train})
    print(f"symbind data: train={len(SB_train)} eval_raw={len(SB_eval_raw)} depths={SYMBIND_DEPTHS}")

    SB_Xd_t, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in SB_train])
    SB_X_dt = SB_Xd_t[L_D]
    SB_Xd_e, _ = states_and_top_tok(model_d, tokenizer_d, [L_D], [p["ids_d"] for p in SB_eval_raw])
    SB_X_de_raw = SB_Xd_e[L_D]
    SB_donor_ok = sb_gen_score(model_d, tokenizer_d, L_D, SB_eval_raw, "ids_d", vecs=None)
    SB_keep = [i for i, ok in enumerate(SB_donor_ok) if ok]
    SB_eval = [SB_eval_raw[i] for i in SB_keep]; SB_X_de = SB_X_de_raw[SB_keep]
    print(f"  donor solves {len(SB_eval)}/{len(SB_eval_raw)} (decoded label)")

    SB_Xr_t, _ = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in SB_train])
    SB_X_rt = SB_Xr_t[L_R]
    SB_Xr_e, _ = states_and_top_tok(model_r, tokenizer_r, [L_R], [p["ids_r"] for p in SB_eval])
    SB_X_re = SB_Xr_e[L_R]
    SB_solvable = sb_gen_score(model_r, tokenizer_r, L_R, SB_eval, "ids_r", vecs=None)
    SB_unsolv = [i for i, ok in enumerate(SB_solvable) if not ok]
    SB_by_depth = {d: [i for i, p in enumerate(SB_eval) if p["depth"] == d] for d in SYMBIND_DEPTHS}
    SB_unsolv_by_depth = {d: [i for i in idxs if not SB_solvable[i]] for d, idxs in SB_by_depth.items()}
    print("  recipient native by depth:",
          {d: fmt(wilson_bools([SB_solvable[i] for i in idxs])) for d, idxs in SB_by_depth.items() if idxs})
    print("  unsolvable counts by depth:", {d: len(v) for d, v in SB_unsolv_by_depth.items()})
    del SB_Xd_t, SB_Xd_e, SB_Xr_t, SB_Xr_e, SB_X_de_raw

    # -------- reconstruction + task maps (same channel battery as arithmetic) --------
    SB_mu_d, SB_mu_r, SB_Wr = fit_ridge(SB_X_dt, SB_X_rt)
    SB_mu_dd, SB_mu_rd = SB_mu_d.to(DEVICE), SB_mu_r.to(DEVICE)
    def SB_map_recon(xd): return (xd.to(DEVICE) - SB_mu_dd) @ SB_Wr.to(DEVICE) + SB_mu_rd

    SB_task_maps = []
    model_r.requires_grad_(False)
    try:
        for seed in SYMBIND_SEEDS:
            torch.manual_seed(seed); random.seed(seed)
            W = SB_Wr.clone().to(DEVICE).requires_grad_(True)
            b = SB_mu_r.clone().to(DEVICE).requires_grad_(True)
            opt = torch.optim.Adam([W, b], lr=1e-3)
            handle = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
            idx = list(range(len(SB_train)))
            try:
                for ep in range(TASK_EPOCHS):
                    random.Random(seed*100+ep).shuffle(idx)
                    for s in range(0, len(idx), ARITH_BATCH):
                        sub = idx[s:s+ARITH_BATCH]
                        _graft["vec"] = (SB_X_dt[sub].to(DEVICE) - SB_mu_dd) @ W + b
                        ids, m = left_pad([SB_train[k]["ids_r"] for k in sub], tokenizer_r.pad_token_id)
                        lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                        tgt = torch.tensor([SB_train[k]["tok_r"] for k in sub], device=DEVICE)
                        loss = F.cross_entropy(lg, tgt)
                        opt.zero_grad(); loss.backward(); opt.step()
            finally:
                handle.remove(); _graft["vec"] = None
            SB_task_maps.append((W.detach(), b.detach()))
            print(f"  symbind seed {seed}: final batch CE {loss.item():.3f}")
    finally:
        model_r.requires_grad_(True)

    def SB_map_task(i):
        W, b = SB_task_maps[i]
        return lambda xd: (xd.to(DEVICE) - SB_mu_dd) @ W + b
    def SB_confer(vecs, idxs):
        return sb_gen_score(model_r, tokenizer_r, L_R, [SB_eval[j] for j in idxs], "ids_r",
                            vecs=list(vecs.detach().float().cpu()))

    SB_out = {"task": "symbolic_binding_chains", "labels": SB_LABELS,
              "layers": {"donor": L_D, "recipient": L_R},
              "train_n": len(SB_train), "eval_raw_n": len(SB_eval_raw),
              "eval_donor_solved_n": len(SB_eval), "unsolv_n": len(SB_unsolv),
              "native_all_donor_solved": fmt(wilson_bools(SB_solvable)), "by_depth": {}}
    if SB_unsolv:
        g = torch.Generator().manual_seed(0)
        sh = torch.randperm(len(SB_unsolv), generator=g)
        sh_idx = [SB_unsolv[int(k)] for k in sh]
        SB_out["unsolv_all"] = {
            "recon": fmt(wilson_bools(SB_confer(SB_map_recon(SB_X_de[SB_unsolv]), SB_unsolv))),
            "task":  fmt(wilson_bools(SB_confer(SB_map_task(0)(SB_X_de[SB_unsolv]), SB_unsolv))),
            "shuffle": fmt(wilson_bools(SB_confer(SB_map_task(0)(SB_X_de[sh_idx]), SB_unsolv))),
            "selfgraft": fmt(wilson_bools(SB_confer(SB_X_re[SB_unsolv].to(DEVICE), SB_unsolv))),
        }
    for d in SYMBIND_DEPTHS:
        idxs = SB_by_depth.get(d, []); uidx = SB_unsolv_by_depth.get(d, [])
        if not idxs: continue
        ys = torch.tensor([SB_eval[i]["label"] for i in idxs])
        row = {"n_donor_solved": len(idxs), "n_unsolv": len(uidx),
               "native": fmt(wilson_bools([SB_solvable[i] for i in idxs])),
               "donor_probe_label": probe_split_acc(SB_X_de[idxs], ys, nclass=len(SB_LABELS)),
               "native_recipient_probe_label": probe_split_acc(SB_X_re[idxs], ys, nclass=len(SB_LABELS)),
               "stitched_probe_label": probe_split_acc(
                   SB_map_task(0)(SB_X_de[idxs]).detach().cpu(), ys, nclass=len(SB_LABELS))}
        if uidx:
            gd = torch.Generator().manual_seed(d)
            shd = torch.randperm(len(uidx), generator=gd)
            shd_idx = [uidx[int(k)] for k in shd]
            row.update({
                "recon_unsolv":   fmt(wilson_bools(SB_confer(SB_map_recon(SB_X_de[uidx]), uidx))),
                "task_unsolv":    fmt(wilson_bools(SB_confer(SB_map_task(0)(SB_X_de[uidx]), uidx))),
                "shuffle_unsolv": fmt(wilson_bools(SB_confer(SB_map_task(0)(SB_X_de[shd_idx]), uidx))),
            })
        SB_out["by_depth"][f"depth{d}"] = row
    RESULTS["symbolic_binding_chains"] = SB_out
    print("EXP8 symbolic binding chains:", json.dumps(SB_out, indent=2))
else:
    print("EXP8 skipped (RUN_SYMBIND=False).")

symbind data: train=1500 eval_raw=1050 depths=[1, 3, 5]
  donor solves 604/1050 (decoded label)
  recipient native by depth: {1: '0.439 [0.387, 0.492]', 3: '0.503 [0.431, 0.574]', 5: '0.494 [0.386, 0.602]'}
  unsolvable counts by depth: {1: 192, 3: 91, 5: 40}
  symbind seed 0: final batch CE 0.049
EXP8 symbolic binding chains: {
  "task": "symbolic_binding_chains",
  "labels": [
    "red",
    "blue",
    "green",
    "yellow",
    "black",
    "white",
    "purple",
    "silver",
    "gold",
    "brown"
  ],
  "layers": {
    "donor": 23,
    "recipient": 20
  },
  "train_n": 1500,
  "eval_raw_n": 1050,
  "eval_donor_solved_n": 604,
  "unsolv_n": 323,
  "native_all_donor_solved": "0.465 [0.426, 0.505]",
  "by_depth": {
    "depth1": {
      "n_donor_solved": 342,
      "n_unsolv": 192,
      "native": "0.439 [0.387, 0.492]",
      "donor_probe_label": "1.000 [0.978, 1.000]",
      "native_recipient_probe_label": "0.626 [0.551, 0.695]",
      "stitched_probe_label": "0.977 [0.941, 0.99

In [20]:
# === CELL SAVE: concentrate everything and write crossfamily_results_<donor>.json ===
keys = [k for k in ("setup", "layer_derivation", "arithmetic_core", "freevec_ceiling",
                    "nonlinear_task_map", "transcription_probe", "answer_ablation",
                    "recipient_feature_delta", "symbolic_binding_chains") if k in RESULTS]
out = {k: RESULTS[k] for k in keys}
out["_config"] = {"donor_arm": DONOR_ARM, "donor": MODEL_D, "recipient": MODEL_R,
                  "L_D": L_D, "L_R": L_R, "smoke_test": SMOKE_TEST,
                  "task_seeds": TASK_SEEDS, "task_epochs": TASK_EPOCHS,
                  "scoring": "decoded-answer only (full-answer string + leading digit); "
                             "no token id compared across families"}
fname = f"crossfamily_results_{MODEL_D.split('/')[-1]}.json"
with open(fname, "w") as f: json.dump(out, f, indent=2)
print("="*72); print("ALL RESULTS (point [95% Wilson CI])"); print("="*72)
print(json.dumps(out, indent=2))
print("saved", fname, "->  DOWNLOAD before terminating the pod")

ALL RESULTS (point [95% Wilson CI])
{
  "setup": {
    "donor": "Qwen/Qwen2.5-7B",
    "recipient": "google/gemma-2-2b",
    "L_D": 23,
    "L_R": 20,
    "d_donor": 3584,
    "d_recipient": 2304,
    "n_train": 3000,
    "n_eval_donor_solved": 508,
    "n_unsolvable": 439,
    "n_solvable": 69,
    "donor_full_on_all_eval": "0.254 [0.235, 0.274]",
    "recipient_native_full_donor_solved": "0.136 [0.109, 0.168]",
    "recipient_native_lead_donor_solved": "0.604 [0.561, 0.646]",
    "recipient_native_lead_on_unsolv": "0.542 [0.495, 0.588]",
    "donor_leading_digit_probe_L23": "0.606 [0.545, 0.664]",
    "_note": "all bins/metrics are decoded-answer based; no token id is compared across families"
  },
  "layer_derivation": {
    "recipient_layer": 20,
    "by_donor_layer": {
      "L13": {
        "donor_probe_leading_digit": "0.358 [0.302, 0.419]",
        "task_confer_unsolv": {
          "full": "0.155 [0.124, 0.192]",
          "lead": "0.809 [0.769, 0.843]"
        },
        "fina